# Steady-State Simulation: Dispatch and Load Flow Analysis

## Objective
This notebook evaluates the impact of new generation on system operation and power flows.

**Project**: 39 Bus New England System - 2  
**Study Case**: Study Cases 1. Power Flow

### Key Objectives:
- Evaluate the impact of new generation on system operation and power flows
- Run load flow simulations for Base Case and New Generation Case
- Extract and analyze: Bus voltage magnitudes and angles, Active and reactive power flows, Generator power outputs
- Create comprehensive visualizations

---

## Step 1: Access PowerFactory

First, we need to set up the Python environment to access DIgSILENT PowerFactory. This involves:
1. Adding PowerFactory to the system PATH
2. Adding PowerFactory's Python library to sys.path
3. Importing the powerfactory module


In [1]:
import os
os.environ["PATH"] = r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2" + os.environ["PATH"]

import sys
sys.path.append(r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9")

# Import powerfactory
import powerfactory as pf
app = pf.GetApplication()  # Get the application


## Step 2: Access and Activate Project

Now we activate the PowerFactory project. This step:
- Gets the current user
- Activates the specific project we want to work with
- Retrieves the active project object for further operations


In [ ]:
user = app.GetCurrentUser()
project = app.ActivateProject("39 Bus New England System - 2")  # Activate the desired project
prj = app.GetActiveProject()

print(f"Project activated: {prj.loc_name}")


## Step 2.5: Activate Study Case

We need to activate the correct study case. This ensures we're working with the right power flow configuration.


In [ ]:
# Try to activate the study case "Study Cases 1. Power Flow"
try:
    study_cases = prj.GetContents('*.IntCase')
    for sc in study_cases:
        if '1. Power Flow' in sc.loc_name or 'Power Flow' in sc.loc_name:
            sc.Activate()
            print(f"Study case activated: {sc.loc_name}")
            break
except:
    print("Note: Using default/active study case")


## Step 3: Get All Relevant Objects

Before running the load flow, we need to collect all the system components we'll be analyzing:
- **Buses (ElmTerm)**: Network nodes where voltages are measured
- **Generators (ElmSym)**: Synchronous generators producing power
- **Lines (ElmLne)**: Transmission lines carrying power between buses

We create dictionaries to easily access these objects by name.


In [ ]:
# Create bus dictionary
buses = app.GetCalcRelevantObjects('*.ElmTerm')
bus_dict = {}
for bus in buses:
    bus_dict[bus.loc_name] = bus

# Create generator dictionary
generators = app.GetCalcRelevantObjects('*.ElmSym')
gen_dict = {}
for gen in generators:
    gen_dict[gen.loc_name] = gen

# Create line dictionary
lines = app.GetCalcRelevantObjects('*.ElmLne')
line_dict = {}
for line in lines:
    line_dict[line.loc_name] = line

print(f"Found {len(bus_dict)} buses, {len(gen_dict)} generators, and {len(line_dict)} lines")


## Step 4: Run Load Flow Analysis - Base Case

The load flow calculation solves the power system equations to determine:
- Bus voltages (magnitude and angle)
- Power flows in lines
- Generator outputs
- System losses

We use a balanced 3-phase calculation (`iopt_net = 0`).


In [ ]:
print("\n=== Running Load Flow Analysis - Base Case ===")
app.ResetCalculation()

# Get load flow calculation command
loadflow = app.GetFromStudyCase('ComLdf')
loadflow.iopt_net = 0  # Balanced 3-phase calculation
loadflow.Execute()

print("Load flow calculation completed for Base Case")


## Step 5: Extract Base Case Results

After the load flow calculation, we extract the results:
- **Bus voltages**: Magnitude (p.u.) and angle (degrees)
- **Generator outputs**: Active power (MW) and reactive power (Mvar)
- **Line flows**: Active and reactive power at both ends of each line

These results are stored in dictionaries for easy access and export.


In [ ]:
base_case_results = {}

# Extract bus voltage magnitudes and angles
for bus_name, bus in bus_dict.items():
    v_mag = bus.GetAttribute('m:u')  # Voltage magnitude in p.u.
    v_angle = bus.GetAttribute('m:phiu')  # Voltage angle in degrees
    base_case_results[bus_name] = {
        'voltage_mag': v_mag,
        'voltage_angle': v_angle
    }

# Extract generator power outputs
gen_results_base = {}
for gen_name, gen in gen_dict.items():
    p_gen = gen.GetAttribute('m:P:bus1')  # Active power in MW
    q_gen = gen.GetAttribute('m:Q:bus1')  # Reactive power in Mvar
    gen_results_base[gen_name] = {
        'P_MW': p_gen,
        'Q_Mvar': q_gen
    }

# Extract line power flows
line_results_base = {}
for line_name, line in line_dict.items():
    p_from = line.GetAttribute('m:P:bus1')  # Active power from bus
    q_from = line.GetAttribute('m:Q:bus1')  # Reactive power from bus
    p_to = line.GetAttribute('m:P:bus2')  # Active power to bus
    q_to = line.GetAttribute('m:Q:bus2')  # Reactive power to bus
    line_results_base[line_name] = {
        'P_from_MW': p_from,
        'Q_from_Mvar': q_from,
        'P_to_MW': p_to,
        'Q_to_Mvar': q_to
    }

print(f"Extracted results for {len(base_case_results)} buses, {len(gen_results_base)} generators, and {len(line_results_base)} lines")


## Step 6: Export Base Case Results to CSV

We export the results to CSV files for:
- **Documentation**: Permanent record of the analysis
- **Further analysis**: Can be imported into Excel, Python, or other tools
- **Visualization**: The CSV files will be loaded back for plotting

We create separate CSV files for buses, generators, and lines.


In [ ]:
import csv
import os

# Get current notebook directory
script_dir = os.getcwd()

# Export bus results
bus_csv_path = os.path.join(script_dir, 'base_case_bus_results.csv')
with open(bus_csv_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Bus Name', 'Voltage Magnitude (p.u.)', 'Voltage Angle (degrees)'])
    for bus_name, results in base_case_results.items():
        writer.writerow([bus_name, results['voltage_mag'], results['voltage_angle']])

print(f"Base case bus results exported to: base_case_bus_results.csv")

# Export generator results
gen_csv_path = os.path.join(script_dir, 'base_case_generator_results.csv')
with open(gen_csv_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Generator Name', 'Active Power (MW)', 'Reactive Power (Mvar)'])
    for gen_name, results in gen_results_base.items():
        writer.writerow([gen_name, results['P_MW'], results['Q_Mvar']])

print(f"Base case generator results exported to: base_case_generator_results.csv")

# Export line results
line_csv_path = os.path.join(script_dir, 'base_case_line_results.csv')
with open(line_csv_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Line Name', 'P From (MW)', 'Q From (Mvar)', 'P To (MW)', 'Q To (Mvar)'])
    for line_name, results in line_results_base.items():
        writer.writerow([line_name, results['P_from_MW'], results['Q_from_Mvar'], 
                        results['P_to_MW'], results['Q_to_Mvar']])

print(f"Base case line results exported to: base_case_line_results.csv")


In [ ]:
# NOTE: This section should be modified based on how new generation is added
# For now, this is a placeholder structure
print("\n=== New Generation Case ===")
print("NOTE: Modify this section to add new generation units to the system")
print("After adding new generation, re-run load flow and compare results")


## Step 8: Load CSV Data and Create Visualizations

Now we create comprehensive visualizations to understand the load flow results:

1. **Static plots (PNG)**: High-quality plots for reports and presentations
2. **Interactive plots (HTML)**: Interactive Bokeh plots for detailed exploration

We'll create:
- Bus voltage magnitude and angle plots
- Generator power output comparisons
- Line power flow analysis


In [ ]:
print("\n=== Creating Visualizations ===")

try:
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from bokeh.plotting import figure, output_file, save
    from bokeh.models import ColumnDataSource, HoverTool
    from bokeh.layouts import gridplot
    
    # Set style
    sns.set_style("whitegrid")
    plt.rcParams['figure.figsize'] = (12, 8)
    
    # Load CSV data
    bus_df = pd.read_csv(bus_csv_path)
    gen_df = pd.read_csv(gen_csv_path)
    line_df = pd.read_csv(line_csv_path)
    
    # 1. Bus Voltage Magnitude Bar Plot
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Voltage magnitude
    bus_df_sorted = bus_df.sort_values('Voltage Magnitude (p.u.)', ascending=False).head(20)
    axes[0, 0].barh(range(len(bus_df_sorted)), bus_df_sorted['Voltage Magnitude (p.u.)'], color='steelblue')
    axes[0, 0].set_yticks(range(len(bus_df_sorted)))
    axes[0, 0].set_yticklabels(bus_df_sorted['Bus Name'], fontsize=8)
    axes[0, 0].set_xlabel('Voltage Magnitude (p.u.)', fontsize=10)
    axes[0, 0].set_title('Top 20 Bus Voltage Magnitudes', fontsize=12, fontweight='bold')
    axes[0, 0].axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='Nominal')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3, axis='x')
    
    # Voltage angle
    axes[0, 1].scatter(bus_df['Voltage Angle (degrees)'], bus_df['Voltage Magnitude (p.u.)'], 
                      alpha=0.6, s=50, color='coral')
    axes[0, 1].set_xlabel('Voltage Angle (degrees)', fontsize=10)
    axes[0, 1].set_ylabel('Voltage Magnitude (p.u.)', fontsize=10)
    axes[0, 1].set_title('Voltage Magnitude vs Angle', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Generator Power Output
    gen_df_sorted = gen_df.sort_values('Active Power (MW)', ascending=False)
    axes[1, 0].bar(range(len(gen_df_sorted)), gen_df_sorted['Active Power (MW)'], 
                   color='green', alpha=0.7, label='Active Power')
    ax2 = axes[1, 0].twinx()
    ax2.bar(range(len(gen_df_sorted)), gen_df_sorted['Reactive Power (Mvar)'], 
            color='orange', alpha=0.7, label='Reactive Power', width=0.6)
    axes[1, 0].set_xticks(range(len(gen_df_sorted)))
    axes[1, 0].set_xticklabels(gen_df_sorted['Generator Name'], rotation=45, ha='right', fontsize=8)
    axes[1, 0].set_ylabel('Active Power (MW)', fontsize=10, color='green')
    ax2.set_ylabel('Reactive Power (Mvar)', fontsize=10, color='orange')
    axes[1, 0].set_title('Generator Power Outputs', fontsize=12, fontweight='bold')
    axes[1, 0].legend(loc='upper left')
    ax2.legend(loc='upper right')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Line Power Flows
    line_df['Total P Flow'] = abs(line_df['P From (MW)']) + abs(line_df['P To (MW)'])
    line_df_sorted = line_df.sort_values('Total P Flow', ascending=False).head(15)
    axes[1, 1].barh(range(len(line_df_sorted)), line_df_sorted['Total P Flow'], color='purple', alpha=0.7)
    axes[1, 1].set_yticks(range(len(line_df_sorted)))
    axes[1, 1].set_yticklabels(line_df_sorted['Line Name'], fontsize=8)
    axes[1, 1].set_xlabel('Total Active Power Flow (MW)', fontsize=10)
    axes[1, 1].set_title('Top 15 Line Power Flows', fontsize=12, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plot_path = os.path.join(script_dir, 'load_flow_analysis_plots.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Static plots saved to: load_flow_analysis_plots.png")
    
except ImportError as e:
    print(f"Note: Visualization libraries not available: {e}")
    print("Install required packages: pip install matplotlib seaborn pandas bokeh")
except Exception as e:
    print(f"Note: Error creating visualizations: {e}")


## Step 8.1: Create Interactive Bokeh Plots

Interactive plots allow you to:
- Zoom and pan for detailed exploration
- Hover over data points to see exact values
- Save customized views

These HTML files can be opened in any web browser.


In [ ]:
# 2. Interactive Bokeh Plot - Bus Voltages
try:
    output_file(os.path.join(script_dir, 'load_flow_interactive.html'))
    
    # Bus voltage interactive plot
    p1 = figure(width=800, height=400, title="Bus Voltage Magnitudes (Interactive)", 
               x_axis_label="Bus Index", y_axis_label="Voltage Magnitude (p.u.)",
               tools="pan,wheel_zoom,box_zoom,reset,hover,save")
    
    source = ColumnDataSource(data=dict(
        x=list(range(len(bus_df))),
        y=bus_df['Voltage Magnitude (p.u.)'],
        bus_names=bus_df['Bus Name'],
        angles=bus_df['Voltage Angle (degrees)']
    ))
    
    p1.circle('x', 'y', size=8, source=source, color='steelblue', alpha=0.6)
    p1.line([0, len(bus_df)], [1.0, 1.0], color='red', line_dash='dashed', line_width=2, legend_label='Nominal (1.0 p.u.)')
    
    hover = p1.select_one(HoverTool)
    hover.tooltips = [("Bus", "@bus_names"), ("Voltage", "@y{0.000} p.u."), ("Angle", "@angles{0.0}°")]
    
    # Generator power interactive plot
    p2 = figure(width=800, height=400, title="Generator Power Outputs (Interactive)",
               x_axis_label="Generator", y_axis_label="Power (MW/Mvar)",
               tools="pan,wheel_zoom,box_zoom,reset,hover,save")
    
    gen_source = ColumnDataSource(data=dict(
        x=list(range(len(gen_df))),
        p_mw=gen_df['Active Power (MW)'],
        q_mvar=gen_df['Reactive Power (Mvar)'],
        gen_names=gen_df['Generator Name']
    ))
    
    p2.vbar(x='x', top='p_mw', width=0.5, source=gen_source, color='green', alpha=0.7, legend_label='Active Power (MW)')
    p2.line('x', 'q_mvar', source=gen_source, color='orange', line_width=3, legend_label='Reactive Power (Mvar)')
    
    hover2 = p2.select_one(HoverTool)
    hover2.tooltips = [("Generator", "@gen_names"), ("P (MW)", "@p_mw{0.0}"), ("Q (Mvar)", "@q_mvar{0.0}")]
    
    # Combine plots
    grid = gridplot([[p1], [p2]], toolbar_location='right')
    save(grid)
    print(f"Interactive plots saved to: load_flow_interactive.html")
except Exception as e:
    print(f"Note: Bokeh interactive plot creation failed: {e}")
    print("Install bokeh for interactive plots: pip install bokeh")


In [ ]:
app.ResetCalculation()

print("\n=== Load Flow Analysis completed successfully ===")
print(f"Results saved in: {script_dir}")
